# Импорт

In [4]:
import os
from dotenv import load_dotenv

from pathlib import Path

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import PointStruct, Filter
from fastembed import TextEmbedding, SparseTextEmbedding

from load_fixtures import load_all_fixtures

import uuid

# Настройки

In [ ]:
project_dir = Path().cwd().parent
data_dir = project_dir / "data"

load_dotenv(project_dir)


namespace_name = os.environ["NAMESPACE_UUID"]
NAMESPACE = uuid.UUID(namespace_name)


# Qdrant

## Подключение

In [6]:
client = QdrantClient(url="http://localhost:6333")

## Создание коллекций

In [7]:
collections = ["cards", "community", "faq", "rules"]

In [8]:
for collection_name in collections:
    client.delete_collection(collection_name)

In [9]:
for collection_name in collections:
    if not client.collection_exists(collection_name=collection_name):
        client.create_collection(
            collection_name=collection_name,
            vectors_config={
                "dense": models.VectorParams(
                    size=1024,
                    distance=models.Distance.COSINE
                )
            },
            sparse_vectors_config={
                "sparse": models.SparseVectorParams(modifier=models.Modifier.IDF)
            }
        )
        print(f"{collection_name}: Коллекция создана")
    else:
        print(f"{collection_name}: Коллекция существует")

cards: Коллекция создана
community: Коллекция создана
faq: Коллекция создана
rules: Коллекция создана


## Загрузка данных

In [10]:
dense_model_name = "intfloat/multilingual-e5-large"
sparse_model_name = "Qdrant/bm25"

dense_embedder = TextEmbedding(dense_model_name, cache_dir=str(project_dir / "models" / dense_model_name))
sparse_embedder = SparseTextEmbedding(sparse_model_name, cache_dir=str(project_dir / "models" / sparse_model_name))

C:\Users\PC\AppData\Local\Temp\ipykernel_15700\2934588083.py:4: UserWarning: The model intfloat/multilingual-e5-large now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  dense_embedder = TextEmbedding(dense_model_name, cache_dir=str(project_dir / "models" / dense_model_name))


In [12]:
fixtures = load_all_fixtures(data_dir)
fixtures

{'rules': [Chunk(id='r001', text='1.1 Мана. Мана — это ресурс, который игрок тратит для розыгрыша карт и активации способностей. Мана восстанавливается в начале каждого хода до текущего лимита маны игрока.', source_type='official', collection='rules', metadata={'section_path': '1.1', 'text': '1.1 Мана. Мана — это ресурс, который игрок тратит для розыгрыша карт и активации способностей. Мана восстанавливается в начале каждого хода до текущего лимита маны игрока.', 'source_type': 'official', 'doc_version': '1.0', 'referenced_sections': [], 'id': 'r001'}),
  Chunk(id='r002', text='1.2 Разряд. Разряд — это единовременная трата всей доступной маны для получения усиленного эффекта карты. После разряда лимит маны не восстанавливается до начала следующего хода.', source_type='official', collection='rules', metadata={'section_path': '1.2', 'text': '1.2 Разряд. Разряд — это единовременная трата всей доступной маны для получения усиленного эффекта карты. После разряда лимит маны не восстанавливае

In [13]:
def embed_dense(texts) -> list:
    return [v.tolist() 
                for v in dense_embedder.embed(
                    list("passage: " + text for text in texts)
            )]

def embed_sparse(texts):
    return [models.SparseVector(
                indices=s.indices.tolist(),
                values=s.values.tolist()
                )
            for s in sparse_embedder.embed(texts)]

def embed_hybrid(chunk) -> PointStruct:
    return PointStruct(
                id=str(uuid.uuid5(NAMESPACE, chunk.id)),
                vector={
                    "dense": embed_dense([chunk.text])[0],
                    "sparse": embed_sparse([chunk.text])[0],
                },
                payload=chunk.metadata
            )

In [14]:
for collection_name in collections:
    points = (
        embed_hybrid(chunk)
        for chunk in fixtures[collection_name]
    )
    client.upload_points(
        collection_name=collection_name,
        points=points,
        batch_size=25
    )

# Тестирование

In [15]:
from rag.retrieval import Retriever

retriever = Retriever()

E:\SSAU_code\berserk-rag\rag\retrieval.py:27: UserWarning: The model intfloat/multilingual-e5-large now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  self.dense_embedder = TextEmbedding(


In [16]:
lst = await retriever.retrieve(query='нельзя блокировать без Дозора', top_k=5)
lst

[RetrievedChunk(chunk=Chunk(id='r006', text='3.2 Засада. Существо со способностью Засада не может быть заблокировано существами, у которых нет способности Дозор, вне зависимости от их характеристик.', source_type='official', collection='rules', metadata={'section_path': '3.2', 'text': '3.2 Засада. Существо со способностью Засада не может быть заблокировано существами, у которых нет способности Дозор, вне зависимости от их характеристик.', 'source_type': 'official', 'doc_version': '1.0', 'referenced_sections': [], 'id': 'r006'}), score=0.7384865283966064, stage='reranked'),
 RetrievedChunk(chunk=Chunk(id='c002', text='Тень Засады. Засада. Тень Засады не может быть заблокирована, пока у противника нет существ со способностью Дозор.', source_type='official', collection='cards', metadata={'name': 'Тень Засады', 'type': 'существо', 'cost': 2, 'text': 'Тень Засады. Засада. Тень Засады не может быть заблокирована, пока у противника нет существ со способностью Дозор.', 'set': 'Базовый выпуск',